In [23]:
# ============================================================
# EXPERIMENT A
# BALANCED ACCURACY OF INDIVIDUAL TARGETED-SMOTE BASE MODELS
# ============================================================

import pandas as pd
import joblib
import time

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = PROJECT_DIR / "dataset" / "cleaned"
MODELS_DIR = PROJECT_DIR / "models"

print("=" * 70)
print("EXPERIMENT A: INDIVIDUAL TARGETED-SMOTE MODEL EVALUATION")
print("=" * 70)

# ============================================================
# LOAD TEST DATA
# ============================================================

print("\nLoading test data...")

X_test = pd.read_parquet(
    DATASET_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    DATASET_DIR / "y_test.parquet"
).squeeze()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# ============================================================
# LOAD MODELS
# ============================================================

model_paths = {
    "Random Forest": MODELS_DIR / "random_forest_targeted_smote.pkl",
    "Extra Trees": MODELS_DIR / "extra_trees_targeted_smote.pkl",
    "XGBoost": MODELS_DIR / "xgboost_targeted_smote.pkl"
}

models = {}

print("\nChecking saved models...")

for name, path in model_paths.items():

    if path.exists():
        models[name] = joblib.load(path)
        print(f"✓ {name} loaded")
    else:
        print(f"✗ {name} NOT FOUND")
        print(f"  Expected: {path}")

# ============================================================
# EVALUATE MODELS
# ============================================================

results = []

for name, model in models.items():

    print("\n" + "=" * 70)
    print(f"EVALUATING: {name.upper()}")
    print("=" * 70)

    start = time.time()

    y_pred = model.predict(X_test)

    prediction_time = time.time() - start

    accuracy = accuracy_score(y_test, y_pred)

    balanced_acc = balanced_accuracy_score(
        y_test,
        y_pred
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(f"\nAccuracy           : {accuracy:.6f}")
    print(f"Weighted F1        : {weighted_f1:.6f}")
    print(f"Macro Precision    : {macro_precision:.6f}")
    print(f"Macro Recall       : {macro_recall:.6f}")
    print(f"Macro F1           : {macro_f1:.6f}")
    print(f"Balanced Accuracy  : {balanced_acc:.6f}")
    print(f"Balanced Accuracy  : {balanced_acc * 100:.4f}%")
    print(f"Prediction Time    : {prediction_time:.2f} seconds")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Weighted F1": weighted_f1,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Balanced Accuracy": balanced_acc,
        "Balanced Accuracy (%)": balanced_acc * 100,
        "Prediction Time (sec)": prediction_time
    })

# ============================================================
# FINAL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Balanced Accuracy",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("EXPERIMENT A - FINAL COMPARISON")
print("=" * 70)

print(results_df.to_string(index=False))

# ============================================================
# COMPARE WITH STACKING
# ============================================================

STACKING_BALANCED_ACCURACY = 0.857707

print("\n" + "=" * 70)
print("COMPARISON WITH FINAL TARGETED-SMOTE STACKING")
print("=" * 70)

best_model = results_df.iloc[0]

print(f"\nBest Individual Model : {best_model['Model']}")
print(
    f"Best Balanced Accuracy: "
    f"{best_model['Balanced Accuracy (%)']:.4f}%"
)

print(
    f"\nStacking Balanced Accuracy: "
    f"{STACKING_BALANCED_ACCURACY * 100:.4f}%"
)

difference = (
    best_model["Balanced Accuracy"]
    - STACKING_BALANCED_ACCURACY
) * 100

print(f"Difference: {difference:+.4f} percentage points")

if difference > 0:
    print(
        "\n✓ IMPORTANT: The best individual model has HIGHER "
        "Balanced Accuracy than the stacking ensemble."
    )
    print(
        "This suggests that the current meta-learner may be reducing "
        "minority-class performance."
    )
else:
    print(
        "\n✓ The stacking ensemble still has the best Balanced Accuracy."
    )

# ============================================================
# SAVE RESULTS
# ============================================================

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

save_path = RESULTS_DIR / "Experiment_A_Targeted_SMOTE_Base_Model_Comparison.csv"

results_df.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)
print(save_path)

EXPERIMENT A: INDIVIDUAL TARGETED-SMOTE MODEL EVALUATION

Loading test data...
X_test shape: (397985, 30)
y_test shape: (397985,)

Checking saved models...
✓ Random Forest loaded
✓ Extra Trees loaded
✓ XGBoost loaded

EVALUATING: RANDOM FOREST

Accuracy           : 0.996580
Weighted F1        : 0.996677
Macro Precision    : 0.835847
Macro Recall       : 0.856682
Macro F1           : 0.836709
Balanced Accuracy  : 0.856682
Balanced Accuracy  : 85.6682%
Prediction Time    : 6.12 seconds

EVALUATING: EXTRA TREES

Accuracy           : 0.996615
Weighted F1        : 0.996709
Macro Precision    : 0.844530
Macro Recall       : 0.859396
Macro F1           : 0.842482
Balanced Accuracy  : 0.859396
Balanced Accuracy  : 85.9396%
Prediction Time    : 5.26 seconds

EVALUATING: XGBOOST

Accuracy           : 0.996653
Weighted F1        : 0.996934
Macro Precision    : 0.805907
Macro Recall       : 0.894318
Macro F1           : 0.820981
Balanced Accuracy  : 0.894318
Balanced Accuracy  : 89.4318%
Predictio

In [ ]:
# ============================================================
# EXPERIMENT B
# META-LEARNER OPTIMIZATION FOR TARGETED-SMOTE STACKING
# ============================================================

import numpy as np
import pandas as pd
import joblib
import time

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)


# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = PROJECT_DIR / "dataset" / "cleaned"

MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. DEFINE DATA PATHS
# ============================================================

TRAIN_STACKING_PATH = (
    DATASET_DIR /
    "stacking_train_features_targeted_smote.npy"
)

TEST_STACKING_PATH = (
    DATASET_DIR /
    "stacking_test_features_targeted_smote.npy"
)

Y_TRAIN_PATH = (
    DATASET_DIR /
    "y_train_targeted_smote.parquet"
)

Y_TEST_PATH = (
    DATASET_DIR /
    "y_test.parquet"
)


# ============================================================
# 3. START EXPERIMENT
# ============================================================

print("=" * 70)
print("EXPERIMENT B: META-LEARNER OPTIMIZATION")
print("=" * 70)


# ============================================================
# 4. CHECK REQUIRED FILES
# ============================================================

print("\nChecking required files...\n")

required_files = {
    "Training stacking features": TRAIN_STACKING_PATH,
    "Test stacking features": TEST_STACKING_PATH,
    "Targeted SMOTE training labels": Y_TRAIN_PATH,
    "Test labels": Y_TEST_PATH
}

for name, path in required_files.items():

    if path.exists():

        print(f"✓ {name}")

    else:

        print(f"✗ MISSING: {name}")
        print(f"  Expected path: {path}")

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )


# ============================================================
# 5. LOAD STACKING FEATURES
# ============================================================

print("\n" + "=" * 70)
print("LOADING STACKING FEATURES")
print("=" * 70)

start_loading = time.time()


X_meta_train = np.load(
    TRAIN_STACKING_PATH
)

X_meta_test = np.load(
    TEST_STACKING_PATH
)


y_meta_train = pd.read_parquet(
    Y_TRAIN_PATH
).squeeze()


y_test = pd.read_parquet(
    Y_TEST_PATH
).squeeze()


loading_time = time.time() - start_loading


print("\n✓ Data loaded successfully")

print(
    f"\nMeta training features : "
    f"{X_meta_train.shape}"
)

print(
    f"Meta test features     : "
    f"{X_meta_test.shape}"
)

print(
    f"Meta training labels   : "
    f"{y_meta_train.shape}"
)

print(
    f"Test labels            : "
    f"{y_test.shape}"
)

print(
    f"Loading time           : "
    f"{loading_time:.2f} seconds"
)


# ============================================================
# 6. VALIDATE DATA
# ============================================================

print("\n" + "=" * 70)
print("VALIDATING STACKING DATA")
print("=" * 70)


if len(X_meta_train) != len(y_meta_train):

    raise ValueError(
        "Mismatch between X_meta_train and y_meta_train."
    )


if len(X_meta_test) != len(y_test):

    raise ValueError(
        "Mismatch between X_meta_test and y_test."
    )


print(
    f"\nTraining samples: "
    f"{len(X_meta_train)}"
)

print(
    f"Test samples    : "
    f"{len(X_meta_test)}"
)

print(
    f"Training classes: "
    f"{y_meta_train.nunique()}"
)

print(
    f"Test classes    : "
    f"{y_test.nunique()}"
)


# ============================================================
# 7. DEFINE META-LEARNERS
# ============================================================

print("\n" + "=" * 70)
print("DEFINING META-LEARNERS")
print("=" * 70)


meta_models = {

    # --------------------------------------------------------
    # META MODEL 1
    # CURRENT TYPE OF META-LEARNER
    # --------------------------------------------------------

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver="lbfgs"
    ),


    # --------------------------------------------------------
    # META MODEL 2
    # --------------------------------------------------------

    "Random Forest Meta": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),


    # --------------------------------------------------------
    # META MODEL 3
    # --------------------------------------------------------

    "Extra Trees Meta": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),


    # --------------------------------------------------------
    # META MODEL 4
    # --------------------------------------------------------

    "XGBoost Meta": XGBClassifier(
        objective="multi:softprob",
        num_class=15,

        n_estimators=200,

        max_depth=6,

        learning_rate=0.1,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="mlogloss",

        tree_method="hist",

        random_state=42
    )
}


print("\nMeta-learners to evaluate:")

for name in meta_models.keys():

    print(f"• {name}")


# ============================================================
# 8. TRAIN AND EVALUATE META-LEARNERS
# ============================================================

results = {}

trained_meta_models = {}


for name, model in meta_models.items():

    print("\n" + "=" * 70)
    print(
        f"TRAINING META-LEARNER: "
        f"{name.upper()}"
    )
    print("=" * 70)


    # ========================================================
    # TRAIN MODEL
    # ========================================================

    print("\nTraining started...")

    start_train = time.time()


    model.fit(
        X_meta_train,
        y_meta_train
    )


    training_time = (
        time.time() - start_train
    )


    print("\n✓ Training completed")

    print(
        f"Training Time: "
        f"{training_time / 60:.2f} minutes"
    )


    # ========================================================
    # PREDICT
    # ========================================================

    print("\nGenerating predictions...")

    start_predict = time.time()


    y_pred = model.predict(
        X_meta_test
    )


    prediction_time = (
        time.time() - start_predict
    )


    print("✓ Predictions completed")


    # ========================================================
    # CALCULATE METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )


    macro_precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )


    macro_recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )


    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )


    balanced_acc = balanced_accuracy_score(
        y_test,
        y_pred
    )


    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("\n" + "-" * 70)
    print(
        f"RESULTS: {name.upper()}"
    )
    print("-" * 70)


    print(
        f"Accuracy           : "
        f"{accuracy:.6f}"
    )

    print(
        f"Weighted F1        : "
        f"{weighted_f1:.6f}"
    )

    print(
        f"Macro Precision    : "
        f"{macro_precision:.6f}"
    )

    print(
        f"Macro Recall       : "
        f"{macro_recall:.6f}"
    )

    print(
        f"Macro F1           : "
        f"{macro_f1:.6f}"
    )

    print(
        f"Balanced Accuracy  : "
        f"{balanced_acc:.6f}"
    )

    print(
        f"Balanced Accuracy  : "
        f"{balanced_acc * 100:.4f}%"
    )

    print(
        f"Prediction Time    : "
        f"{prediction_time:.2f} seconds"
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    results[name] = {

        "Meta Learner": name,

        "Accuracy": accuracy,

        "Weighted F1": weighted_f1,

        "Macro Precision": macro_precision,

        "Macro Recall": macro_recall,

        "Macro F1": macro_f1,

        "Balanced Accuracy": balanced_acc,

        "Balanced Accuracy (%)": (
            balanced_acc * 100
        ),

        "Training Time (min)": (
            training_time / 60
        ),

        "Prediction Time (sec)": (
            prediction_time
        )
    }


    # ========================================================
    # STORE TRAINED MODEL
    # ========================================================

    trained_meta_models[name] = model


# ============================================================
# 9. CREATE FINAL COMPARISON TABLE
# ============================================================

print("\n" + "=" * 70)
print("CREATING META-LEARNER COMPARISON")
print("=" * 70)


results_df = pd.DataFrame(
    list(results.values())
)


results_df = results_df.sort_values(
    by="Balanced Accuracy",
    ascending=False
).reset_index(
    drop=True
)


print("\n" + "=" * 70)
print("EXPERIMENT B - META-LEARNER COMPARISON")
print("=" * 70)


print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 10. COMPARE WITH EXISTING MODELS
# ============================================================

CURRENT_STACKING_BALANCED_ACC = 0.857707

BEST_XGB_BALANCED_ACC = 0.894318


best_meta = results_df.iloc[0]


best_meta_name = (
    best_meta["Meta Learner"]
)


best_meta_balanced_acc = (
    best_meta["Balanced Accuracy"]
)


print("\n" + "=" * 70)
print("FINAL BALANCED ACCURACY COMPARISON")
print("=" * 70)


print(
    f"\nOriginal Logistic Stacking : "
    f"{CURRENT_STACKING_BALANCED_ACC * 100:.4f}%"
)


print(
    f"Best Individual XGBoost    : "
    f"{BEST_XGB_BALANCED_ACC * 100:.4f}%"
)


print(
    f"Best Experiment B Stacking : "
    f"{best_meta_balanced_acc * 100:.4f}%"
)


print(
    f"\nBest Meta-Learner: "
    f"{best_meta_name}"
)


# ============================================================
# 11. COMPARE BEST STACKING WITH XGBOOST
# ============================================================

difference_vs_xgb = (
    best_meta_balanced_acc
    - BEST_XGB_BALANCED_ACC
) * 100


difference_vs_old_stacking = (
    best_meta_balanced_acc
    - CURRENT_STACKING_BALANCED_ACC
) * 100


print("\nImprovement over old stacking:")

print(
    f"{difference_vs_old_stacking:+.4f} "
    f"percentage points"
)


print("\nDifference vs individual XGBoost:")

print(
    f"{difference_vs_xgb:+.4f} "
    f"percentage points"
)


if difference_vs_xgb > 0:

    print("\n✓ SUCCESS")

    print(
        "The optimized stacking ensemble "
        "has beaten the best individual XGBoost model."
    )


elif difference_vs_xgb == 0:

    print("\n✓ TIE")

    print(
        "The optimized stacking ensemble "
        "matches the individual XGBoost model."
    )


else:

    print("\n⚠ XGBOOST REMAINS STRONGER")

    print(
        "The best individual XGBoost model "
        "still has higher Balanced Accuracy."
    )


# ============================================================
# 12. SAVE BEST META-LEARNER
# ============================================================

print("\n" + "=" * 70)
print("SAVING BEST META-LEARNER")
print("=" * 70)


best_model = (
    trained_meta_models[
        best_meta_name
    ]
)


best_model_path = (
    MODELS_DIR /
    "best_meta_learner_experiment_B.pkl"
)


joblib.dump(
    best_model,
    best_model_path
)


print("\n✓ Best meta-learner saved")

print(
    best_model_path
)


print(
    f"\nSelected meta-learner: "
    f"{best_meta_name}"
)


# ============================================================
# 13. SAVE COMPARISON RESULTS
# ============================================================

results_path = (
    RESULTS_DIR /
    "Experiment_B_Meta_Learner_Comparison.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


print("\n✓ Experiment B results saved")

print(
    results_path
)


# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EXPERIMENT B COMPLETED")
print("=" * 70)


print(
    f"\nBest Meta-Learner       : "
    f"{best_meta_name}"
)


print(
    f"Best Balanced Accuracy  : "
    f"{best_meta_balanced_acc * 100:.4f}%"
)


print(
    f"Best Individual XGBoost : "
    f"{BEST_XGB_BALANCED_ACC * 100:.4f}%"


    


print(
    f"Original Stacking       : "
    f"{CURRENT_STACKING_BALANCED_ACC * 100:.4f}%"
)

EXPERIMENT B: META-LEARNER OPTIMIZATION

Checking required files...

✓ Training stacking features
✓ Test stacking features
✓ Targeted SMOTE training labels
✓ Test labels

LOADING STACKING FEATURES

✓ Data loaded successfully

Meta training features : (1993626, 45)
Meta test features     : (397985, 45)
Meta training labels   : (1993626,)
Test labels            : (397985,)
Loading time           : 1.23 seconds

VALIDATING STACKING DATA

Training samples: 1993626
Test samples    : 397985
Training classes: 15
Test classes    : 15

DEFINING META-LEARNERS

Meta-learners to evaluate:
• Logistic Regression
• Random Forest Meta
• Extra Trees Meta
• XGBoost Meta

TRAINING META-LEARNER: LOGISTIC REGRESSION

Training started...

✓ Training completed
Training Time: 0.16 minutes

Generating predictions...
✓ Predictions completed

----------------------------------------------------------------------
RESULTS: LOGISTIC REGRESSION
----------------------------------------------------------------------
A